# Prediccion de Resultados de Combates de Boxeo
## Entrenamiento y optimizacion de Modelos de Machine Learning - Entrega Final

**Indice**

* **1. Abstracto: motivacion y audiencia**
* **2. Problema a resolver**
* **3. Carga y revision de datos**
    * 3.1 Carga del dataset
    * 3.2 Deteccion de outliers
    * 3.3 Revision de valores faltantes
* **4. Analisis Exploratorio de Datos (EDA)**
    * 4.1 Variables numericas
    * 4.2 Variables categoricas
* **5. Imputacion de valores faltantes**
* **6. Ingenieria de atributos**
    * 6.1 Nuevas variables derivadas del estilo de pelea
    * 6.2 Encoding de variables categoricas
    * 6.3 Normalizacion / Estandarizacion
* **7. Modelado**
    * 7.1 Separacion train/test
    * 7.2 Validacion cruzada de modelos base
    * 7.3 Optimizacion de hiperparametros (GridSearchCV)
    * 7.4 Curvas de aprendizaje
    * 7.5 Importancia de atributos (SHAP)
    * 7.6 Ensamble de modelos (Voting Classifier)
* **8. Seleccion del modelo final y conclusiones**


## 1. Abstracto: motivacion y audiencia

El boxeo profesional es un deporte donde el resultado de un combate depende de
multiples factores fisicos, tecnicos y de experiencia de cada peleador: alcance,
altura, edad, record de victorias/derrotas, porcentaje de nocauts, estilo de pelea
(ortodoxo o zurdo), etc.

**Motivacion:** promotoras de boxeo, casas de apuestas deportivas, analistas y
entrenadores se benefician de poder estimar, antes de un combate, la probabilidad
de que un boxeador gane en funcion de sus atributos fisicos y su historial frente
al de su rival. Esto permite armar carteleras mas competitivas, calibrar lineas de
apuestas y disenar estrategias de entrenamiento enfocadas en las debilidades
detectadas.

**Audiencia:**
* Promotoras y matchmakers que deciden que enfrentamientos organizar.
* Casas de apuestas / analistas deportivos que necesitan estimar probabilidades.
* Entrenadores y equipos de un boxeador, para entender que atributos pesan mas
  a la hora de definir un combate.
* Medios y aficionados interesados en pronosticos basados en datos.

En este notebook se retoma el trabajo de la entrega anterior (analisis exploratorio
de un dataset de combates de boxeo) y se avanza hacia la construccion de modelos
de **Machine Learning** que permitan predecir el resultado de un combate.


## 2. Preguntas / Problema que buscamos resolver

**Problema principal (clasificacion binaria):**

> Dado un enfrentamiento entre el "Boxeador A" y el "Boxeador B", a partir de sus
> atributos fisicos y su historial de carrera, **predecir si el Boxeador A
> gana el combate** (`gano_A = 1`) o no (`gano_A = 0`).

Este es un problema de **clasificacion binaria**, donde la variable objetivo es
`gano_A`.

**Preguntas secundarias que tambien exploraremos:**

* Que atributos (alcance, edad, porcentaje de KO, record, etc.) tienen mayor
  influencia en el resultado de un combate?
* Existen diferencias relevantes segun el estilo de pelea (ortodoxo vs zurdo)?
* Que modelo de Machine Learning ofrece el mejor desempeno (medido con AUC y
  accuracy) para este problema?


## 3. Carga y revision de datos

### 3.1 Carga del dataset

Dado que no contamos con un dataset publico unico y limpio de enfrentamientos de
boxeo con todos los atributos que necesitamos, **generamos un dataset sintetico
pero realista** de combates, a partir de distribuciones tipicas de atributos de
boxeadores profesionales (alcance, altura, edad, record, porcentaje de KO, etc.).

Esto nos permite tener control total sobre el tamano del dataset, introducir
valores faltantes y outliers de forma controlada (igual que ocurriria con datos
reales scrapeados de fuentes como BoxRec), y manteniendo la misma estructura de
trabajo que tendriamos con datos reales.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from collections import Counter

from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                               GradientBoostingClassifier, ExtraTreesClassifier,
                               VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, GridSearchCV, cross_val_score,
                                      StratifiedKFold, learning_curve)
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, roc_curve
from xgboost import XGBClassifier
import shap

sns.set(style='white', context='notebook', palette='deep')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Generacion del dataset sintetico de combates de boxeo
n = 1500

def generar_boxeador(n):
    altura = np.random.normal(178, 7, n)            # cm
    alcance = altura + np.random.normal(2, 4, n)     # cm, correlacionado con la altura
    edad = np.random.normal(28, 4, n).clip(18, 45)
    peleas = np.random.poisson(18, n).clip(1, 60)
    victorias = (peleas * np.random.beta(6, 2, n)).round().clip(0)
    derrotas = (peleas - victorias).clip(0)
    ko_pct = np.random.beta(2, 3, n) * 100            # % de victorias por KO
    estilo = np.random.choice(['ortodoxo', 'zurdo'], size=n, p=[0.8, 0.2])
    return pd.DataFrame({
        'altura': altura, 'alcance': alcance, 'edad': edad,
        'peleas': peleas, 'victorias': victorias, 'derrotas': derrotas,
        'ko_pct': ko_pct, 'estilo': estilo
    })

boxer_A = generar_boxeador(n).add_suffix('_A')
boxer_B = generar_boxeador(n).add_suffix('_B')

data = pd.concat([boxer_A, boxer_B], axis=1)

# Peso de la categoria del combate
divisiones = ['Peso Mosca', 'Peso Pluma', 'Peso Ligero', 'Peso Welter',
               'Peso Mediano', 'Peso Pesado']
data['division'] = np.random.choice(divisiones, size=n)

# Resultado: probabilidad de que gane A en funcion del diferencial de atributos
ratio_victorias_A = data['victorias_A'] / data['peleas_A']
ratio_victorias_B = data['victorias_B'] / data['peleas_B']

score = (
    0.6 * (data['alcance_A'] - data['alcance_B']) / 10
    + 2.5 * (ratio_victorias_A - ratio_victorias_B)
    + 0.03 * (data['ko_pct_A'] - data['ko_pct_B'])
    - 0.08 * (data['edad_A'] - data['edad_B'])
    + np.random.normal(0, 1.2, n)   # ruido / azar del combate
)
prob_gana_A = 1 / (1 + np.exp(-score))
data['gano_A'] = (np.random.rand(n) < prob_gana_A).astype(int)

# Introducimos valores faltantes de forma controlada (como ocurriria con datos reales)
for col in ['alcance_A', 'alcance_B', 'ko_pct_A', 'ko_pct_B', 'estilo_A', 'estilo_B']:
    mask = np.random.rand(n) < 0.04
    data.loc[mask, col] = np.nan

# Introducimos algunos outliers de edad (datos mal cargados)
outlier_idx = np.random.choice(n, size=5, replace=False)
data.loc[outlier_idx, 'edad_A'] = data.loc[outlier_idx, 'edad_A'] * 2.5

print(data.shape)
data.head()

### 3.2 Deteccion de outliers

Revisamos las variables numericas con boxplots para detectar valores extremos
(por ejemplo, edades cargadas incorrectamente).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(y=data['edad_A'], ax=axes[0]).set_title('Edad Boxeador A')
sns.boxplot(y=data['alcance_A'], ax=axes[1]).set_title('Alcance Boxeador A')
sns.boxplot(y=data['ko_pct_A'], ax=axes[2]).set_title('% KO Boxeador A')
plt.tight_layout()

In [ ]:
# Eliminamos outliers claros de edad (valores fisicamente imposibles para un boxeador activo)
outliers = data[(data['edad_A'] > 50) | (data['edad_B'] > 50)].index
print(f"Outliers detectados y eliminados: {len(outliers)}")
data = data.drop(outliers).reset_index(drop=True)

### 3.3 Revision de valores faltantes

Verificamos la cantidad de valores nulos por columna.


In [ ]:
nulos = data.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print(nulos)

plt.figure(figsize=(8, 4))
sns.heatmap(data.isnull(), cbar=False, yticklabels=False)
plt.title('Mapa de valores faltantes')

## 4. Analisis Exploratorio de Datos (EDA)

### 4.1 Variables numericas

Analizamos la distribucion de las variables numericas y su relacion con la
variable objetivo `gano_A`.


In [ ]:
data.describe()

In [ ]:
# Distribucion de la variable objetivo
plt.figure(figsize=(5, 4))
sns.countplot(x='gano_A', data=data)
plt.title('Distribucion de la variable objetivo (gano_A)')
plt.show()

data['gano_A'].value_counts(normalize=True)

In [ ]:
# Matriz de correlacion entre variables numericas y la variable objetivo
plt.figure(figsize=(12, 9))
num_cols = data.select_dtypes(include=[np.number]).columns
sns.heatmap(data[num_cols].corr(), annot=False, cmap='coolwarm', center=0)
plt.title('Matriz de correlacion - variables numericas')

In [ ]:
# Diferencia de alcance vs resultado
plt.figure(figsize=(6, 4))
data['dif_alcance'] = data['alcance_A'] - data['alcance_B']
sns.violinplot(x='gano_A', y='dif_alcance', data=data)
plt.title('Diferencia de alcance (A - B) segun resultado')

In [ ]:
# Ratio de victorias historicas vs resultado
data['ratio_victorias_A'] = data['victorias_A'] / data['peleas_A']
data['ratio_victorias_B'] = data['victorias_B'] / data['peleas_B']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(x='gano_A', y='ratio_victorias_A', data=data, ax=axes[0])
axes[0].set_title('Ratio de victorias de A segun resultado')
sns.boxplot(x='gano_A', y='ratio_victorias_B', data=data, ax=axes[1])
axes[1].set_title('Ratio de victorias de B segun resultado')
plt.tight_layout()

### 4.2 Variables categoricas

Exploramos el estilo de pelea y la division de peso.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x='estilo_A', hue='gano_A', data=data, ax=axes[0])
axes[0].set_title('Estilo de pelea de A vs resultado')

sns.countplot(y='division', hue='gano_A', data=data, ax=axes[1])
axes[1].set_title('Division de peso vs resultado')

plt.tight_layout()

## 5. Imputacion de valores faltantes

* **Variables numericas** (`alcance_A`, `alcance_B`, `ko_pct_A`, `ko_pct_B`):
  se imputan con la **mediana** de cada columna, ya que es una medida robusta
  frente a outliers.
* **Variables categoricas** (`estilo_A`, `estilo_B`): se imputan con la
  **moda** (valor mas frecuente), que en boxeo es `ortodoxo`.


In [ ]:
for col in ['alcance_A', 'alcance_B', 'ko_pct_A', 'ko_pct_B']:
    data[col] = data[col].fillna(data[col].median())

for col in ['estilo_A', 'estilo_B']:
    data[col] = data[col].fillna(data[col].mode()[0])

print("Valores faltantes restantes:", data.isnull().sum().sum())

## 6. Ingenieria de atributos

### 6.1 Nuevas variables derivadas

Creamos variables que capturan **diferencias relativas** entre ambos boxeadores,
que suelen ser mas informativas que los valores absolutos a la hora de predecir
el resultado de un enfrentamiento:

* `dif_alcance`: diferencia de alcance (A - B), ya calculada en el EDA.
* `dif_altura`: diferencia de altura.
* `dif_edad`: diferencia de edad.
* `dif_ko_pct`: diferencia de porcentaje de KO.
* `dif_ratio_victorias`: diferencia de ratio de victorias historicas.
* `experiencia_A` / `experiencia_B`: numero total de peleas, como proxy de
  experiencia.
* `mismo_estilo`: indica si ambos boxeadores comparten el mismo estilo de pelea.


In [ ]:
data['dif_altura'] = data['altura_A'] - data['altura_B']
data['dif_edad'] = data['edad_A'] - data['edad_B']
data['dif_ko_pct'] = data['ko_pct_A'] - data['ko_pct_B']
data['dif_ratio_victorias'] = data['ratio_victorias_A'] - data['ratio_victorias_B']
data['experiencia_A'] = data['peleas_A']
data['experiencia_B'] = data['peleas_B']
data['mismo_estilo'] = (data['estilo_A'] == data['estilo_B']).astype(int)

data.head()

### 6.2 Encoding de variables categoricas

* `estilo_A` y `estilo_B`: encoding binario (1 = zurdo, 0 = ortodoxo).
* `division`: one-hot encoding, ya que no tiene un orden natural relevante
  para el modelo.


In [ ]:
data['estilo_A'] = data['estilo_A'].map({'ortodoxo': 0, 'zurdo': 1})
data['estilo_B'] = data['estilo_B'].map({'ortodoxo': 0, 'zurdo': 1})

data = pd.get_dummies(data, columns=['division'], drop_first=True)

data.head()

### 6.3 Normalizacion / Estandarizacion

Estandarizamos las variables numericas continuas con `StandardScaler` para que
todas tengan media 0 y desvio estandar 1. Esto es especialmente importante para
modelos sensibles a la escala como **SVM**, **KNN** y **regresion logistica**.

La estandarizacion se ajusta (`fit`) **solo sobre el conjunto de entrenamiento**
para evitar fuga de informacion (*data leakage*) hacia el conjunto de test.


In [ ]:
target = 'gano_A'
y = data[target]
X = data.drop(columns=[target])

print("Variables predictoras:", X.shape[1])
print("Observaciones:", X.shape[0])

## 7. Modelado

### 7.1 Separacion train / test

Separamos el dataset en conjunto de entrenamiento (80%) y testeo (20%), de forma
estratificada para mantener la proporcion de la variable objetivo.


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
num_cols_to_scale = X_train.select_dtypes(include=[np.number]).columns

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols_to_scale] = scaler.fit_transform(X_train[num_cols_to_scale])
X_test_scaled[num_cols_to_scale] = scaler.transform(X_test[num_cols_to_scale])

print("Train:", X_train_scaled.shape, " Test:", X_test_scaled.shape)

### 7.2 Validacion cruzada de modelos base

Entrenamos y comparamos varios modelos de clasificacion mediante **validacion
cruzada estratificada (k=5)**, usando **AUC (ROC-AUC)** y **accuracy** como
metricas principales:

* Regresion Logistica
* K-Nearest Neighbors
* Support Vector Machine (SVC)
* Decision Tree
* Random Forest
* Extra Trees
* Gradient Boosting
* AdaBoost
* XGBoost


In [ ]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

modelos = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(),
    'SVC': SVC(probability=True, random_state=RANDOM_STATE),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE),
    'ExtraTrees': ExtraTreesClassifier(random_state=RANDOM_STATE),
    'GradientBoosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'AdaBoost': AdaBoostClassifier(random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE)
}

resultados_cv = []
for nombre, modelo in modelos.items():
    auc_scores = cross_val_score(modelo, X_train_scaled, Y_train, cv=kfold, scoring='roc_auc')
    acc_scores = cross_val_score(modelo, X_train_scaled, Y_train, cv=kfold, scoring='accuracy')
    resultados_cv.append({
        'modelo': nombre,
        'auc_mean': auc_scores.mean(), 'auc_std': auc_scores.std(),
        'acc_mean': acc_scores.mean(), 'acc_std': acc_scores.std()
    })

cv_df = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
cv_df

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x='auc_mean', y='modelo', data=cv_df, xerr=cv_df['auc_std'])
plt.title('AUC promedio (validacion cruzada, k=5) por modelo')
plt.xlabel('AUC')
plt.xlim(0.5, 1.0)
plt.show()

### 7.3 Optimizacion de hiperparametros (GridSearchCV)

A partir de los resultados de validacion cruzada, seleccionamos los **dos
modelos con mejor desempeno** (por ejemplo Random Forest y Gradient Boosting,
aunque el resultado puede variar segun la semilla aleatoria) y optimizamos sus
hiperparametros mediante `GridSearchCV`, usando AUC como metrica de seleccion.


In [ ]:
# --- Random Forest ---
rf_param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [None, 4, 8, 12],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=rf_param_grid, cv=kfold, scoring='roc_auc', n_jobs=-1, verbose=0
)
rf_grid.fit(X_train_scaled, Y_train)

print("Mejor AUC (CV) Random Forest:", rf_grid.best_score_)
print("Mejores hiperparametros:", rf_grid.best_params_)
rf_best = rf_grid.best_estimator_

In [ ]:
# --- Gradient Boosting ---
gb_param_grid = {
    'n_estimators': [100, 200, 400],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_grid=gb_param_grid, cv=kfold, scoring='roc_auc', n_jobs=-1, verbose=0
)
gb_grid.fit(X_train_scaled, Y_train)

print("Mejor AUC (CV) Gradient Boosting:", gb_grid.best_score_)
print("Mejores hiperparametros:", gb_grid.best_params_)
gb_best = gb_grid.best_estimator_

In [ ]:
# --- XGBoost ---
xgb_param_grid = {
    'n_estimators': [100, 200, 400],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE),
    param_grid=xgb_param_grid, cv=kfold, scoring='roc_auc', n_jobs=-1, verbose=0
)
xgb_grid.fit(X_train_scaled, Y_train)

print("Mejor AUC (CV) XGBoost:", xgb_grid.best_score_)
print("Mejores hiperparametros:", xgb_grid.best_params_)
xgb_best = xgb_grid.best_estimator_

### 7.4 Curvas de aprendizaje

Las curvas de aprendizaje permiten detectar problemas de **overfitting** o
**underfitting** observando como evolucionan el error de entrenamiento y de
validacion a medida que aumenta el tamano del conjunto de entrenamiento.


In [ ]:
def plot_learning_curve(estimator, title, X, y, cv, n_jobs=-1,
                         train_sizes=np.linspace(.1, 1.0, 5)):
    plt.figure(figsize=(7, 5))
    plt.title(title)
    plt.xlabel("Ejemplos de entrenamiento")
    plt.ylabel("AUC")

    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes, scoring='roc_auc')
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)

    plt.grid()
    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                      train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                      test_scores_mean + test_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Score entrenamiento")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Score validacion cruzada")
    plt.legend(loc="best")
    return plt

plot_learning_curve(rf_best, "Curva de aprendizaje - Random Forest", X_train_scaled, Y_train, cv=kfold)
plot_learning_curve(gb_best, "Curva de aprendizaje - Gradient Boosting", X_train_scaled, Y_train, cv=kfold)
plot_learning_curve(xgb_best, "Curva de aprendizaje - XGBoost", X_train_scaled, Y_train, cv=kfold)
plt.show()

### 7.5 Importancia de atributos (SHAP)

Utilizamos **SHAP (SHapley Additive exPlanations)** sobre el mejor modelo basado
en arboles para entender que atributos del enfrentamiento influyen mas en la
prediccion de victoria del Boxeador A.


In [ ]:
explainer = shap.TreeExplainer(xgb_best)
shap_values = explainer.shap_values(X_test_scaled)

shap.summary_plot(shap_values, X_test_scaled, show=True)

In [ ]:
# Importancia de atributos segun Random Forest, Gradient Boosting y XGBoost
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (nombre, modelo) in zip(axes, [('Random Forest', rf_best),
                                        ('Gradient Boosting', gb_best),
                                        ('XGBoost', xgb_best)]):
    importancias = pd.Series(modelo.feature_importances_, index=X_train_scaled.columns)
    importancias.sort_values(ascending=True).tail(10).plot.barh(ax=ax)
    ax.set_title(f'Importancia de atributos - {nombre}')

plt.tight_layout()
plt.show()

### 7.6 Ensamble de modelos (Voting Classifier)

Combinamos los tres modelos optimizados (Random Forest, Gradient Boosting y
XGBoost) mediante un **VotingClassifier** con votacion *soft*, que promedia las
probabilidades estimadas por cada modelo.


In [ ]:
voting_clf = VotingClassifier(
    estimators=[('rf', rf_best), ('gb', gb_best), ('xgb', xgb_best)],
    voting='soft', n_jobs=-1
)
voting_clf.fit(X_train_scaled, Y_train)

auc_voting_cv = cross_val_score(voting_clf, X_train_scaled, Y_train, cv=kfold, scoring='roc_auc')
print("AUC promedio (CV) - Voting Classifier:", auc_voting_cv.mean())

## 8. Seleccion del modelo final y conclusiones

Evaluamos todos los modelos optimizados sobre el **conjunto de test**, utilizando
**AUC** y **accuracy** como metricas de comparacion, y seleccionamos el modelo con
mejor desempeno general.


In [ ]:
modelos_finales = {
    'Random Forest (optimizado)': rf_best,
    'Gradient Boosting (optimizado)': gb_best,
    'XGBoost (optimizado)': xgb_best,
    'Voting Classifier (ensamble)': voting_clf
}

resultados_test = []
for nombre, modelo in modelos_finales.items():
    y_pred = modelo.predict(X_test_scaled)
    y_proba = modelo.predict_proba(X_test_scaled)[:, 1]
    resultados_test.append({
        'modelo': nombre,
        'AUC_test': roc_auc_score(Y_test, y_proba),
        'Accuracy_test': accuracy_score(Y_test, y_pred)
    })

resultados_test_df = pd.DataFrame(resultados_test).sort_values('AUC_test', ascending=False)
resultados_test_df

In [ ]:
# Curva ROC del mejor modelo
mejor_modelo_nombre = resultados_test_df.iloc[0]['modelo']
mejor_modelo = modelos_finales[mejor_modelo_nombre]

y_proba_best = mejor_modelo.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, _ = roc_curve(Y_test, y_proba_best)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'{mejor_modelo_nombre} (AUC = {roc_auc_score(Y_test, y_proba_best):.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC - Mejor modelo')
plt.legend(loc='lower right')
plt.show()

print(classification_report(Y_test, mejor_modelo.predict(X_test_scaled)))

### Conclusiones

* Se modelo el problema de prediccion del resultado de un combate de boxeo como
  un **problema de clasificacion binaria** (`gano_A`).
* Las variables derivadas de **diferencias entre boxeadores** (alcance, ratio
  de victorias historicas, edad, porcentaje de KO) resultaron entre las mas
  relevantes segun el analisis de importancia de atributos y SHAP.
* Se entrenaron y compararon multiples modelos mediante **validacion cruzada
  estratificada**, y se optimizaron los mejores mediante **GridSearchCV**.
* El **ensamble (Voting Classifier)** combinando Random Forest, Gradient
  Boosting y XGBoost logro un desempeno competitivo (medido en AUC y accuracy),
  consolidando las fortalezas individuales de cada modelo.
* El modelo seleccionado (mostrado como primera fila en la tabla de resultados
  de test) puede ser utilizado por promotoras, casas de apuestas y entrenadores
  como **herramienta de apoyo** para estimar probabilidades de victoria antes
  de un combate, sin reemplazar el analisis experto del cuerpo tecnico.

**Trabajo futuro:** incorporar datos reales de combates (por ejemplo, de
BoxRec), variables adicionales como golpes conectados por round, tiempo desde
el ultimo combate (inactividad) y resultados frente a rivales en comun.
